# Notebook 07 – Exploratory Data Analysis (EDA)

This notebook explores the audited master dataset separately by source. It creates descriptive statistics, distributions, process–response plots, within-source correlations, target eligibility/value-concentration audits and a downloadable report archive.

**Input:** upload `master_experimental_dataset_qc.csv` from the Notebook 06 output ZIP. Do not use the unaudited master for this notebook.

EDA is descriptive. Target-specific eligibility is applied to process–response plots and correlations. Correlations are not causal evidence, and separate studies must not be pooled simply because they share a column name.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import files
from IPython.display import display
import shutil

uploaded = files.upload()
matches = [n for n in uploaded if n.lower().endswith('.csv') and 'master_experimental_dataset_qc' in n.lower()]
if not matches:
    raise FileNotFoundError('Upload master_experimental_dataset_qc.csv from the Notebook 06 outputs.')
df = pd.read_csv(matches[0])
print('Loaded:', matches[0], '| Shape:', df.shape)
if 'source_id' not in df.columns:
    raise ValueError('The audited master file must contain source_id.')
display(df.groupby('source_id').size().rename('records').to_frame())
display(df.head())

## 1. Define source-specific inputs and outcomes

The mapping follows the current audited master schema. Missing columns are skipped rather than guessed.

In [ ]:
SOURCE_MAP = {
 'conventional_laser_welding_dataset': {'inputs':['laser_power_W','welding_speed_mm_s'], 'targets':['weld_bead_width_mm','interface_width_mm','penetration_depth_mm'], 'eligibility':{'weld_bead_width_mm':'eligible_weld_bead_width','interface_width_mm':'eligible_interface_width','penetration_depth_mm':'eligible_penetration_depth_conventional'}},
 'alcu_wobbling_welding_dataset': {'inputs':['laser_power_W','tangential_wobbling_speed_mm_s','wobbling_diameter_mm'], 'targets':['bead_width_face_mm','bead_width_interface_mm','penetration_depth_mm'], 'eligibility':{'bead_width_face_mm':'eligible_bead_width_face_wobble','bead_width_interface_mm':'eligible_bead_width_interface_wobble','penetration_depth_mm':'eligible_penetration_depth_wobble'}},
 'shin_2021_metals_11050831': {'inputs':['laser_power_W','welding_speed_mm_s','reported_heat_input_J_mm'], 'targets':['tensile_shear_strength_MPa','electrical_resistance_microohm','welding_area_mm2'], 'eligibility':{'tensile_shear_strength_MPa':'eligible_tensile_shear_strength','electrical_resistance_microohm':'eligible_electrical_resistance','welding_area_mm2':'eligible_welding_area'}},
 'lee_2022_metals_12111957': {'inputs':['welding_speed_mm_s'], 'targets':['force_average_N_reported'], 'eligibility':{'force_average_N_reported':'eligible_force_average_N'}}
}
for sid in df['source_id'].dropna().unique():
    if sid not in SOURCE_MAP:
        print('No explicit variable map for source:', sid)


## 2. Source-specific descriptive statistics

Summaries are calculated only for numeric columns with observations. They are not pooled across source groups.

In [ ]:
outdir = Path('notebook07_eda_outputs')
figdir = outdir/'figures'
figdir.mkdir(parents=True, exist_ok=True)
summary_frames = []
for sid, cfg in SOURCE_MAP.items():
    g = df.loc[df['source_id'].eq(sid)].copy()
    if g.empty: continue
    cols = [c for c in cfg['inputs'] + cfg['targets'] if c in g.columns]
    numeric = g[cols].apply(pd.to_numeric, errors='coerce')
    desc = numeric.describe().T.reset_index().rename(columns={'index':'variable'})
    desc.insert(0, 'source_id', sid)
    summary_frames.append(desc)
    print(f'\n{sid}: {len(g)} records')
    display(desc)
summary = pd.concat(summary_frames, ignore_index=True) if summary_frames else pd.DataFrame()
summary.to_csv(outdir/'descriptive_statistics_by_source.csv', index=False)


## 3. Distribution plots by source

Each variable is plotted only within its own source group. Missing values remain in the data and are excluded only from the histogram.

In [ ]:
for sid, cfg in SOURCE_MAP.items():
    g = df.loc[df['source_id'].eq(sid)]
    if g.empty: continue
    for col in cfg['inputs'] + cfg['targets']:
        if col not in g.columns: continue
        vals = pd.to_numeric(g[col], errors='coerce').dropna()
        if vals.empty: continue
        fig, ax = plt.subplots(figsize=(6,4))
        ax.hist(vals, bins=min(10, max(3, int(np.sqrt(len(vals))))), edgecolor='black')
        ax.set_title(f'{sid}\n{col}')
        ax.set_xlabel(col); ax.set_ylabel('Record count')
        fig.tight_layout()
        safe = ''.join(ch if ch.isalnum() else '_' for ch in f'{sid}_{col}')
        fig.savefig(figdir/f'distribution_{safe}.png', dpi=160)
        plt.show(); plt.close(fig)


## 4. Target eligibility, value coverage and concentration

This audit checks each target using its own eligibility flag. It reports eligible/missing counts, unique measured values, and the most frequent value. A highly repeated value can indicate a measurement/reporting ceiling or discretisation and should be investigated before modelling; it is not automatically an error.

The conventional all-zero geometry rows are retained in the raw master table but excluded from eligible geometry summaries. Wobble face and interface widths are evaluated separately. Wobble penetration depth is also checked for concentration at a single reported value.

In [ ]:
target_audit_rows = []
for sid, cfg in SOURCE_MAP.items():
    g = df.loc[df["source_id"].eq(sid)].copy()
    if g.empty:
        continue
    for target in cfg["targets"]:
        if target not in g.columns:
            continue
        flag = cfg.get("eligibility", {}).get(target)
        eligible = g.copy()
        if flag and flag in eligible.columns:
            eligible = eligible.loc[eligible[flag].fillna(False).astype(bool)]
        values = pd.to_numeric(eligible[target], errors="coerce").dropna()
        raw_values = pd.to_numeric(g[target], errors="coerce").dropna()
        if values.empty:
            target_audit_rows.append({
                "source_id": sid, "target": target,
                "eligibility_flag": flag or "not_available",
                "raw_nonmissing_n": len(raw_values),
                "eligible_n": 0, "eligible_missing_n": len(eligible),
                "raw_missing_n": len(g) - len(raw_values),
                "excluded_by_flag_n": len(g) - len(eligible),
                "unique_values": 0, "minimum": np.nan, "median": np.nan,
                "maximum": np.nan, "most_frequent_value": np.nan,
                "most_frequent_count": 0, "most_frequent_fraction": np.nan
            })
            continue
        counts = values.value_counts(dropna=True)
        target_audit_rows.append({
            "source_id": sid, "target": target,
            "eligibility_flag": flag or "not_available",
            "raw_nonmissing_n": len(raw_values),
            "eligible_n": len(values),
            "eligible_missing_n": len(eligible) - len(values),\n            "raw_missing_n": len(g) - len(raw_values),\n            "excluded_by_flag_n": len(g) - len(eligible),
            "unique_values": int(values.nunique()),
            "minimum": values.min(), "median": values.median(),
            "maximum": values.max(),
            "most_frequent_value": counts.index[0],
            "most_frequent_count": int(counts.iloc[0]),
            "most_frequent_fraction": float(counts.iloc[0] / len(values))
        })

target_audit = pd.DataFrame(target_audit_rows)
target_audit.to_csv(outdir/"target_eligibility_distribution_audit.csv", index=False)
display(target_audit)

# Highlight targets where one exact value dominates the eligible measurements.
concentration_review = target_audit.loc[
    target_audit["most_frequent_fraction"].ge(0.70)
].copy()
print("Targets with >=70% of eligible records at one exact value:")
display(concentration_review)

# For wobble penetration depth, print exact value frequencies for transparent review.
wobble = df.loc[df["source_id"].eq("alcu_wobbling_welding_dataset")].copy()
if "penetration_depth_mm" in wobble.columns:
    flag = "eligible_penetration_depth_wobble"
    if flag in wobble.columns:
        wobble = wobble.loc[wobble[flag].fillna(False).astype(bool)]
    print("Wobble penetration-depth value counts (eligible records):")
    display(
        pd.to_numeric(wobble["penetration_depth_mm"], errors="coerce")
        .value_counts(dropna=False).sort_index()
        .rename_axis("penetration_depth_mm")
        .to_frame("record_count")
    )

# Separate face and interface width coverage for the wobble source.
for target in ["bead_width_face_mm", "bead_width_interface_mm"]:
    if target in wobble.columns:
        vals = pd.to_numeric(
            df.loc[df["source_id"].eq("alcu_wobbling_welding_dataset"), target],
            errors="coerce"
        ).dropna()
        print(f"{target}: n={len(vals)}, unique={vals.nunique()}, missing={df.loc[df['source_id'].eq('alcu_wobbling_welding_dataset'), target].isna().sum()}")

## 5. Process–response scatter plots

Relationships are plotted within each source. If a target eligibility flag exists, only eligible rows are shown.

In [ ]:
scatter_rows = []
for sid, cfg in SOURCE_MAP.items():
    g = df.loc[df['source_id'].eq(sid)].copy()
    if g.empty: continue
    for xcol in cfg['inputs']:
        for ycol in cfg['targets']:
            if xcol not in g.columns or ycol not in g.columns: continue
            flag = cfg.get('eligibility', {}).get(ycol)
            plot_data = g.copy()
            if flag and flag in plot_data.columns:
                plot_data = plot_data.loc[plot_data[flag].fillna(False).astype(bool)]
            x = pd.to_numeric(plot_data[xcol], errors='coerce')
            y = pd.to_numeric(plot_data[ycol], errors='coerce')
            valid = x.notna() & y.notna()
            if valid.sum() < 2: continue
            fig, ax = plt.subplots(figsize=(6,4))
            ax.scatter(x[valid], y[valid])
            ax.set_title(f'{sid}\n{ycol} vs {xcol} (n={int(valid.sum())})')
            ax.set_xlabel(xcol); ax.set_ylabel(ycol)
            fig.tight_layout()
            safe = ''.join(ch if ch.isalnum() else '_' for ch in f'{sid}_{ycol}_vs_{xcol}')
            fig.savefig(figdir/f'scatter_{safe}.png', dpi=160)
            plt.show(); plt.close(fig)
            scatter_rows.append({'source_id':sid,'input':xcol,'target':ycol,'n_plotted':int(valid.sum()),'eligibility_flag':flag or 'not_available'})
pd.DataFrame(scatter_rows).to_csv(outdir/'scatter_plot_inventory.csv', index=False)


## 6. Eligibility-aware within-source correlation analysis

Pearson and Spearman coefficients are calculated separately within each source. For every process-input/target pair, the target's own eligibility flag is applied before calculating the correlation. This is essential for conventional geometry: the six all-zero records are retained in the master dataset but excluded from eligible geometry correlations.

Input–input correlations remain descriptive of all source records. Target–target correlations use rows eligible for both targets. Pairwise sample counts are saved. Small samples and repeated process conditions can make estimates unstable; correlations are not causal effects.

In [ ]:
corr_rows = []
for sid, cfg in SOURCE_MAP.items():
    g = df.loc[df["source_id"].eq(sid)].copy()
    inputs = [c for c in cfg["inputs"] if c in g.columns]
    targets = [c for c in cfg["targets"] if c in g.columns]
    variables = inputs + targets

    for i, a in enumerate(variables):
        for b in variables[i+1:]:
            pair_data = g[[a, b]].copy()

            # Apply target-specific eligibility whenever a target is part of this pair.
            flags_to_apply = []
            for variable in (a, b):
                flag = cfg.get("eligibility", {}).get(variable)
                if variable in targets and flag and flag in g.columns:
                    flags_to_apply.append(flag)
            for flag in flags_to_apply:
                pair_data = pair_data.loc[g.loc[pair_data.index, flag].fillna(False).astype(bool)]

            pair_data[a] = pd.to_numeric(pair_data[a], errors="coerce")
            pair_data[b] = pd.to_numeric(pair_data[b], errors="coerce")
            pair_data = pair_data.dropna(subset=[a, b])
            n_pair = len(pair_data)
            if n_pair < 3:
                continue

            for method in ["pearson", "spearman"]:
                coefficient = pair_data[a].corr(pair_data[b], method=method)
                if pd.notna(coefficient):
                    corr_rows.append({
                        "source_id": sid,
                        "method": method,
                        "variable_1": a,
                        "variable_2": b,
                        "correlation": coefficient,
                        "n_pairwise": n_pair,
                        "eligibility_filters_applied": ";".join(flags_to_apply) if flags_to_apply else "none"
                    })

corr_table = pd.DataFrame(corr_rows)
corr_table.to_csv(outdir/"correlations_within_source_eligible.csv", index=False)
display(corr_table)

print("Conventional correlations above use eligible target rows only; compare their pairwise n with the full 27-record source count.")

## 7. Repeated-condition and quality-flag review

Repeated conditions are part of experimental design and are not automatically duplicates.

In [ ]:
repeat_summaries = []
grouping = {
 'conventional_laser_welding_dataset':['laser_power_W','welding_speed_mm_s'],
 'alcu_wobbling_welding_dataset':['laser_power_W','tangential_wobbling_speed_mm_s','wobbling_diameter_mm'],
 'shin_2021_metals_11050831':['laser_power_W','welding_speed_mm_s'],
 'lee_2022_metals_12111957':['upper_material','welding_speed_mm_s']
}
for sid, keys in grouping.items():
    g = df.loc[df['source_id'].eq(sid)]
    keys = [k for k in keys if k in g.columns]
    if not keys or g.empty: continue
    counts = g.groupby(keys, dropna=False).size().rename('records_at_condition').reset_index()
    counts.insert(0,'source_id',sid)
    repeat_summaries.append(counts)
repeat_table = pd.concat(repeat_summaries, ignore_index=True) if repeat_summaries else pd.DataFrame()
repeat_table.to_csv(outdir/'records_per_reported_condition.csv', index=False)
display(repeat_table)

flag_cols = [c for c in df.columns if c.endswith('_flag') or c.startswith('eligible_')]
if flag_cols:
    flag_summary = df.groupby('source_id')[flag_cols].agg(lambda s: s.fillna(False).astype(bool).sum())
    flag_summary.to_csv(outdir/'quality_and_eligibility_flag_counts.csv')
    display(flag_summary)


## 9. Export EDA tables, figures and the ML-ready wobble subset

The ZIP contains descriptive tables, correlations, repeat-condition summaries, flag counts, generated figures and `wobbling_ml_dataset.csv`. The audited input file is not modified.This creates the same 29-record, 17-condition dataset used for the initial grouped model comparison. Both face and interface bead-width eligibility flags must be true. No values are imputed. The exported CSV is the direct input to Notebook 08.

In [ ]:
features = [
    'laser_power_W',
    'tangential_wobbling_speed_mm_s',
    'wobbling_diameter_mm'
]
targets = ['bead_width_face_mm', 'bead_width_interface_mm']
required_flags = ['eligible_bead_width_face_wobble', 'eligible_bead_width_interface_wobble']
required = ['source_id'] + features + targets + required_flags
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns for wobble ML subset: {missing}')

wobble_ml = df.loc[df['source_id'].eq('alcu_wobbling_welding_dataset')].copy()
for flag in required_flags:
    wobble_ml = wobble_ml.loc[wobble_ml[flag].fillna(False).astype(bool)]
wobble_ml = wobble_ml.dropna(subset=features + targets).copy()
wobble_ml['condition_group'] = wobble_ml[features].astype(str).agg('_'.join, axis=1)

wobble_ml.to_csv('wobbling_ml_dataset.csv', index=False)
wobble_ml.to_csv(outdir/'wobbling_ml_dataset.csv', index=False)
print('Wobbling ML dataset created')
print('Records:', len(wobble_ml))
print('Unique conditions:', wobble_ml['condition_group'].nunique())
print('Records per condition:')
display(wobble_ml['condition_group'].value_counts().value_counts().sort_index().rename_axis('records_per_condition').to_frame('number_of_conditions'))
print('Missing values:')
display(wobble_ml[features + targets].isna().sum().to_frame('missing_count'))
print('Saved: wobbling_ml_dataset.csv (also included in Notebook 07 ZIP)')


## 8. Export EDA tables and figures

The ZIP contains descriptive tables, correlations, repeat-condition summaries, flag counts and generated figures. The audited input file is not modified.

In [ ]:
(outdir/'README.txt').write_text('All summaries and plots are source-specific and exploratory. Target eligibility flags are applied to target-specific analyses. Concentrated target values require source/measurement review; correlations are not causal. Review source provenance and target definitions before modelling.\n', encoding='utf-8')
zip_path = shutil.make_archive('notebook07_eda_outputs', 'zip', outdir)
print('Created:', zip_path)
files.download(zip_path)
